In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots

include("functions.jl")
Random.seed!(2025)
const bb = 27 
const aa = 38
const N  = 1462439
const I0 = 1
const S0 = 1316195
const n_iter = 1_000_000

Istar_obs = [3, 7, 10, 4, 14, 35, 92, 98, 216, 374, 434, 417, 447, 275, 151, 83, 67, 48, 37, 29, 42, 46, 53, 73, 87, 111, 123, 124, 99, 135, 106, 74, 39, 13]

tau = length(Istar_obs)

model_tag_sym = :sliding
KMAX_UPPER = 30  
KMAX_fixed = 27

# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 1

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER,
            k_max_fixed = KMAX_fixed)
    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [sliding_model] Fitting chain 1 (tau=34)
[ Info: [sliding] iter 1000/1000000 elapsed=3.9s, rate=0.093, mean=[1.275, 0.00026, 1.475], std=[0.2285, 0.000288, 0.0214] [ADAPT]
[ Info: [sliding] iter 2000/1000000 elapsed=6.8s, rate=0.052, mean=[1.727, 0.00018, 1.439], std=[0.4417, 0.000228, 0.0374] [ADAPT]
[ Info: [sliding] iter 3000/1000000 elapsed=8.9s, rate=0.039, mean=[1.918, 0.00014, 1.419], std=[0.4349, 0.000197, 0.0398] [ADAPT]
[ Info: [sliding] iter 4000/1000000 elapsed=11.1s, rate=0.031, mean=[1.983, 0.00013, 1.401], std=[0.3902, 0.000178, 0.0447] [ADAPT]
[ Info: [sliding] iter 5000/1000000 elapsed=13.3s, rate=0.027, mean=[2.028, 0.00012, 1.392], std=[0.3580, 0.000165, 0.0431] [ADAPT]
[ Info: [sliding] iter 6000/1000000 elapsed=15.5s, rate=0.025, mean=[2.058, 0.00011, 1.391], std=[0.3354, 0.000156, 0.0395] [ADAPT]
[ Info: [sliding] iter 7000/1000000 elapsed=17.7s, rate=0.024, mean=[2.102, 0.00011, 1.389], std=[0.3261, 0.000149, 0.0371] [ADAPT]
[ Info: [sliding] iter 8000/10